In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install autogluon.tabular lightgbm scikit-learn scipy -q

import pandas as pd, numpy as np, json
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
from scipy.stats import ks_2samp
from autogluon.tabular import TabularPredictor
import warnings; warnings.filterwarnings('ignore')

FEAT_DIR = '/content/drive/MyDrive/ML competition/feature'
DATA_DIR = '/content/drive/MyDrive/ML competition/competition_data'
CSV_DIR  = '/content/drive/MyDrive/ML competition/csv'

# === 피쳐 구성 (523) ===
feats_1st = pd.read_parquet(f'{FEAT_DIR}/feats_1st.parquet')
v3        = pd.read_parquet(f'{FEAT_DIR}/tx_stack_v3.parquet')
ex_tr = pd.read_parquet(f'{FEAT_DIR}/extra_features_train.parquet')
ex_te = pd.read_parquet(f'{FEAT_DIR}/extra_features_test.parquet')
tm_tr = pd.read_parquet(f'{FEAT_DIR}/timing_train.parquet')
tm_te = pd.read_parquet(f'{FEAT_DIR}/timing_test.parquet')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')

extra = pd.concat([ex_tr, ex_te]); extra.columns = ['ex_'+c for c in extra.columns]
timing = pd.concat([tm_tr, tm_te])
X = feats_1st.join(v3).join(extra).join(timing)
X.columns = [str(c) for c in X.columns]
X = X.loc[:, ~X.columns.duplicated()].fillna(0)
feat = list(X.columns)
tr_ids = y_train['custid'].values; tr_set = set(tr_ids)
test_ids = [c for c in X.index if c not in tr_set]
t = y_train.set_index('custid')['gender'].loc[tr_ids].values
print('전체 피쳐:', len(feat))
'''


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 9.6 MB/s eta 0:00:00
전체 피쳐: 523


"\n# ============================================================\n# STEP 1: Adversarial Validation → drift 피쳐 제거\n# ============================================================\nprint('\n[1/4] KS drift 계산 → 안전 피쳐 457개 확보...')\nis_tr = pd.Series(X.index).isin(tr_set).values\nks_all = {}\nfor c in feat:\n    a = X[c].values[is_tr]; b = X[c].values[~is_tr]\n    ks_all[c] = 0.0 if (np.std(a)<1e-9 and np.std(b)<1e-9) else ks_2samp(a,b).statistic\nks_ser = pd.Series(ks_all)\nsafe = [c for c in feat if ks_ser[c] < 0.1]\nprint(f'안전 피쳐: {len(safe)}개 (제거: {len(feat)-len(safe)}개)')\n\n# Adversarial AUC 검증\nXsafe = X[safe].fillna(0)\ny_adv = (~pd.Series(X.index).isin(tr_set)).astype(int).values\nskf3 = StratifiedKFold(3, shuffle=True, random_state=42)\no_adv = np.zeros(len(y_adv))\nfor tri, vi in skf3.split(Xsafe.values, y_adv):\n    m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,\n        num_leaves=31, n_jobs=-1, verbose=-1)\n    m.fit(Xsafe.values[tri], y_adv[tri])\n    o_adv[vi] 

In [ ]:
# 현재 X 피쳐 구성 확인
print('전체 피쳐 수:', len(feat))
print('\nex_ 로 시작하는 피쳐:', len([c for c in feat if c.startswith('ex_')]))
print('txp_ 로 시작하는 피쳐:', len([c for c in feat if c.startswith('txp_')]))
print('t_ 로 시작하는 피쳐:', len([c for c in feat if c.startswith('t_')]))
print('svd_ 로 시작하는 피쳐:', len([c for c in feat if c.startswith('svd_')]))
print('tfidf_ 로 시작하는 피쳐:', len([c for c in feat if c.startswith('tfidf_')]))
print('cal_ 로 시작하는 피쳐:', len([c for c in feat if c.startswith('cal_')]))
print('\n제거된 10개 (KS>0.1):')
removed = [c for c in feat if ks_ser[c] >= 0.1]
for c in removed: print(f'  KS={ks_ser[c]:.4f}  {c}')

NameError: name 'feat' is not defined

In [ ]:
#1. 523개 피처 KS 분포 확인

# KS 분포 확인 + 임계값별 제거 수
for th in [0.10, 0.07, 0.05, 0.03, 0.01]:
    removed = [c for c in feat if ks_ser[c] >= th]
    print(f'KS>={th}: {len(removed)}개 제거 → {len(feat)-len(removed)}개 남음')

print('\nKS 상위 30 (0.03 이상):')
ks_top = ks_ser[ks_ser >= 0.03].sort_values(ascending=False)
for c, v in ks_top.items():
    print(f'  {v:.4f}  {c}')

KS>=0.1: 10개 제거 → 513개 남음
KS>=0.07: 11개 제거 → 512개 남음
KS>=0.05: 13개 제거 → 510개 남음
KS>=0.03: 21개 제거 → 502개 남음
KS>=0.01: 65개 제거 → 458개 남음

KS 상위 30 (0.03 이상):
  0.9672  ex_te_team_nm_min
  0.5375  ex_te_team_nm_max
  0.4241  ex_te_buyer_nm_min
  0.3240  ex_te_team_nm_std
  0.2643  ex_te_buyer_nm_max
  0.1701  ex_te_team_nm_mean
  0.1591  ex_te_pc_nm_min
  0.1579  ex_te_buyer_str_min
  0.1281  ex_te_pc_nm_max
  0.1056  ex_te_buyer_str_max
  0.0724  ex_te_corner_str_min
  0.0605  ex_te_corner_str_max
  0.0584  ex_te_buyer_nm_std
  0.0469  txp_std
  0.0454  txp_spread
  0.0434  ex_te_corner_str_std
  0.0426  ex_te_buyer_str_std
  0.0390  txp_ax_lo_gap
  0.0368  txp_max
  0.0368  ex_te_pc_nm_std
  0.0327  txp_min


In [ ]:
#2. 523개 피처 -> te 제외(498개)  : 0.55857

# tfidf 제외했을 때 adversarial AUC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def adv_auc(cols):
    Xd = X[cols].fillna(0).values.astype('float32')
    y_adv = (~pd.Series(X.index).isin(tr_set)).astype(int).values
    skf = StratifiedKFold(3, shuffle=True, random_state=42)
    o = np.zeros(len(Xd))
    for tri, vi in skf.split(Xd, y_adv):
        m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
            num_leaves=31, n_jobs=-1, verbose=-1)
        m.fit(Xd[tri], y_adv[tri])
        o[vi] = m.predict_proba(Xd[vi])[:, 1]
    return roc_auc_score(y_adv, o)

# 각 피쳐군 제외 테스트
no_tfidf = [c for c in feat if not c.startswith('tfidf_')]
no_svd   = [c for c in feat if not c.startswith('svd_')]
no_te    = [c for c in feat if 'te_' not in c]
safe10   = [c for c in feat if ks_ser[c] < 0.10]
safe001  = [c for c in feat if ks_ser[c] < 0.01]

print(f'전체 523        : adversarial AUC = 0.99250')
print(f'tfidf 제외({len(no_tfidf)}개): {adv_auc(no_tfidf):.5f}')
print(f'svd  제외({len(no_svd)}개) : {adv_auc(no_svd):.5f}')
print(f'te   제외({len(no_te)}개)  : {adv_auc(no_te):.5f}')
print(f'KS<0.01({len(safe001)}개)  : {adv_auc(safe001):.5f}')

전체 523        : adversarial AUC = 0.99250 (이미 앎)
tfidf 제외(223개): 0.99998
svd  제외(503개) : 0.99994
te   제외(498개)  : 0.55857
KS<0.01(458개)  : 0.53118


In [ ]:
#3.481개 피처로 골라내기.

from google.colab import drive
drive.mount('/content/drive')
!pip install autogluon.tabular lightgbm -q

import pandas as pd, numpy as np, json
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
from autogluon.tabular import TabularPredictor
import warnings; warnings.filterwarnings('ignore')

FEAT_DIR = '/content/drive/MyDrive/ML competition/feature'
DATA_DIR = '/content/drive/MyDrive/ML competition/competition_data'
CSV_DIR  = '/content/drive/MyDrive/ML competition/csv'

# === 데이터 ===
f503 = pd.read_parquet(f'{FEAT_DIR}/features_503.parquet')
if 'custid' in f503.columns: f503 = f503.set_index('custid')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')
tr_ids = y_train['custid'].values; tr_set = set(tr_ids)
test_ids = [c for c in f503.index if c not in tr_set]
t = y_train.set_index('custid')['gender'].loc[tr_ids].values
X = f503.fillna(0); feat = list(X.columns)
print(f'503 피쳐: {len(feat)} | train: {len(tr_ids)} | test: {len(test_ids)}')

print('\n[1/3] 안전 pool 구성...')
is_tr = pd.Series(X.index).isin(tr_set).values
ks_ser = pd.Series({
    c: (0.0 if (np.std(X[c].values[is_tr])<1e-9 and np.std(X[c].values[~is_tr])<1e-9)
        else ks_2samp(X[c].values[is_tr], X[c].values[~is_tr]).statistic)
    for c in feat
})
safe = [c for c in feat if ('te_' not in c) and (ks_ser[c] < 0.05)]
print(f'안전 pool: {len(safe)}개 (adversarial AUC ≈ 0.559 — 이미 확인됨)')

# === importance top-N 스윕 (OOF 최대화 기준) ===
print('\n[2/3] importance top-N 스윕...')
Xtr = X.loc[tr_ids][safe].values.astype(np.float32)
imp = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.04, num_leaves=31,
    subsample=0.8, colsample_bytree=0.55, reg_lambda=2.0,
    n_jobs=-1, max_bin=127, verbose=-1,
    importance_type='gain', random_state=42
).fit(Xtr, t).feature_importances_
ranked = [safe[i] for i in np.argsort(-imp)]

skf = StratifiedKFold(5, shuffle=True, random_state=42)
folds = list(skf.split(Xtr, t))

def oof_topN(N):
    cols = ranked[:N]
    Xc = X.loc[tr_ids][cols].values.astype(np.float32)
    o = np.zeros(len(Xc))
    for tri, vi in folds:
        m = lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.04, num_leaves=31,
            subsample=0.8, colsample_bytree=0.55, reg_lambda=2.0,
            n_jobs=-1, max_bin=127, verbose=-1)
        m.fit(Xc[tri], t[tri]); o[vi] = m.predict_proba(Xc[vi])[:,1]
    return roc_auc_score(t, o)

results = {}
print(f'{"N":>5}  {"OOF":>8}')
for N in [100, 150, 200, 250, 300, 350, 400, len(safe)]:
    auc = oof_topN(N)
    results[N] = auc
    print(f'{N:5d}  {auc:.5f}')

best_N = max(results, key=results.get)
print(f'\nOOF 최고 N: {best_N} ({results[best_N]:.5f})')
print(f'(참고: 172 OOF ≈ 0.72109, 기존250 OOF ≈ 0.72125)')


# multi-seed 안정성 확인
print('\nmulti-seed 검증 (seed 42/7/2024):')
def ms_oof(cols):
    res = []
    for seed in [42, 7, 2024]:
        skf2 = StratifiedKFold(5, shuffle=True, random_state=seed)
        Xd = X.loc[tr_ids][cols].values.astype(np.float32)
        o = np.zeros(len(Xd))
        for tri, vi in skf2.split(Xd, t):
            m = lgb.LGBMClassifier(
                n_estimators=300, learning_rate=0.04, num_leaves=31,
                subsample=0.8, colsample_bytree=0.55, reg_lambda=2.0,
                n_jobs=-1, max_bin=127, verbose=-1)
            m.fit(Xd[tri], t[tri]); o[vi] = m.predict_proba(Xd[vi])[:,1]
        res.append(roc_auc_score(t, o))
    return np.mean(res), np.std(res)

m_, s_ = ms_oof(best_sel)
print(f'attack top{best_N}: mean={m_:.5f} seed_std={s_:.5f}')
print(f'(참고: 172 seed_std=0.00030, 기존250 seed_std=0.00040)')
'''
'''

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
503 피쳐: 503 | train: 30000 | test: 19995

[1/3] 안전 pool 구성...
안전 pool: 481개 (adversarial AUC ≈ 0.559 — 이미 확인됨)

[2/3] importance top-N 스윕...
    N       OOF
  100  0.71886
  150  0.72038
  200  0.72098
  250  0.71945
  300  0.71864
  350  0.71969
  400  0.71862
  481  0.71836

OOF 최고 N: 200 (0.72098)
(참고: 172 OOF ≈ 0.72109, 기존250 OOF ≈ 0.72125)

multi-seed 검증 (seed 42/7/2024):
attack top200: mean=0.71962 seed_std=0.00097
(참고: 172 seed_std=0.00030, 기존250 seed_std=0.00040)


"\n# === AutoGluon 4시간 ===\nprint(f'\n[3/3] AutoGluon 4시간 (K={best_N})...')\nXsel = X[best_sel]\ntrain_data = Xsel.loc[tr_ids].copy(); train_data['gender'] = t\nX_test = Xsel.loc[test_ids]\n\npredictor = TabularPredictor(\n    label='gender', problem_type='binary', eval_metric='roc_auc'\n).fit(\n    train_data,\n    presets='best_quality',\n    time_limit=14400,          # 4시간\n    dynamic_stacking=False,    # DyStack 스킵 (이미 0으로 확인됨)\n    num_stack_levels=0         # 바로 본 학습에 4시간 집중\n)\n\npred = predictor.predict_proba(X_test)[1]\npd.DataFrame({'custid': X_test.index, 'gender': pred}).to_csv(\n    f'{CSV_DIR}/submission_attack.csv', index=False)\nprint('AG OOF:', predictor.leaderboard(silent=True).iloc[0]['score_val'])\nprint('saved submission_attack.csv')\n"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install autogluon.tabular -q

import pandas as pd, numpy as np, json
from autogluon.tabular import TabularPredictor

FEAT_DIR = '/content/drive/MyDrive/ML competition/feature'
DATA_DIR = '/content/drive/MyDrive/ML competition/competition_data'
CSV_DIR  = '/content/drive/MyDrive/ML competition/csv'

# 저장된 피쳐 로드
best_sel = json.load(open(f'{FEAT_DIR}/sel_attack.json'))
print(f'attack 피쳐: {len(best_sel)}개')

f503 = pd.read_parquet(f'{FEAT_DIR}/features_503.parquet')
if 'custid' in f503.columns: f503 = f503.set_index('custid')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')
tr_ids = y_train['custid'].values; tr_set = set(tr_ids)
test_ids = [c for c in f503.index if c not in tr_set]
t = y_train.set_index('custid')['gender'].loc[tr_ids].values
X = f503.fillna(0)

Xsel = X[best_sel]
train_data = Xsel.loc[tr_ids].copy(); train_data['gender'] = t
X_test = Xsel.loc[test_ids]

predictor = TabularPredictor(
    label='gender', problem_type='binary', eval_metric='roc_auc'
).fit(
    train_data,
    presets='best_quality',
    time_limit=3600,
    dynamic_stacking=False,
    num_stack_levels=0
)

pred = predictor.predict_proba(X_test)[1]
pd.DataFrame({'custid': X_test.index, 'gender': pred}).to_csv(
    f'{CSV_DIR}/submission_attack.csv', index=False)
print('AG OOF:', predictor.leaderboard(silent=True).iloc[0]['score_val'])
print('saved submission_attack.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 9.4 MB/s eta 0:00:00
attack 피쳐: 200개


No path specified. Models will be saved in: "AutogluonModels/ag-20260610_041541"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       9.66 GB / 12.67 GB (76.2%)
Disk Space Avail:   65.42 GB / 112.64 GB (58.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "/content/AutogluonModels/ag-20260610_041541"
Train Data Rows:    30000
Train Data Columns: 20

[1000]	valid_set's binary_logloss: 0.543873


	0.7204	 = Validation score   (roc_auc)
	247.97s	 = Training   runtime
	1.97s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 1763.17s of the 1763.15s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
No improvement since epoch 2: early stopping
No improvement since epoch 7: early stopping
No improvement since epoch 2: early stopping
No improvement since epoch 3: early stopping
No improvement since epoch 1: early stopping
No improvement since epoch 2: early stopping
No improvement since epoch 2: early stopping
No improvement since epoch 1: early stopping
	0.6917	 = Validation score   (roc_auc)
	505.29s	 = Training   runtime
	0.83s	 = Validation runtime
Fitting model: CatBoost_r9_BAG_L1 ... Training model for up to 1256.47s of the 1256.44s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential:

[1000]	valid_set's binary_logloss: 0.538935
[1000]	valid_set's binary_logloss: 0.542378
[1000]	valid_set's binary_logloss: 0.533776
[2000]	valid_set's binary_logloss: 0.5336
[3000]	valid_set's binary_logloss: 0.533965
[1000]	valid_set's binary_logloss: 0.544799
[1000]	valid_set's binary_logloss: 0.545758
[1000]	valid_set's binary_logloss: 0.541291
[1000]	valid_set's binary_logloss: 0.542112
[1000]	valid_set's binary_logloss: 0.544998


	0.7248	 = Validation score   (roc_auc)
	370.48s	 = Training   runtime
	3.12s	 = Validation runtime
Fitting model: NeuralNetTorch_r22_BAG_L1 ... Training model for up to 881.04s of the 881.01s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
	0.7203	 = Validation score   (roc_auc)
	324.42s	 = Training   runtime
	1.77s	 = Validation runtime
Fitting model: XGBoost_r33_BAG_L1 ... Training model for up to 554.46s of the 554.43s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
	0.7112	 = Validation score   (roc_auc)
	531.41s	 = Training   runtime
	0.53s	 = Validation runtime
Fitting model: ExtraTrees_r42_BAG_L1 ... Training model for up to 22.11s of the 22.08s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0, mem=0.1/8.5 GB
	0.7028	 = Validation score   (roc_auc)
	149.19

AG OOF: 0.7270042922221079
saved submission_attack.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, json
from scipy.stats import ks_2samp

FEAT_DIR = '/content/drive/MyDrive/ML competition/feature'
DATA_DIR = '/content/drive/MyDrive/ML competition/competition_data'

# === 503 로드 ===
f503 = pd.read_parquet(f'{FEAT_DIR}/features_503.parquet')
if 'custid' in f503.columns: f503 = f503.set_index('custid')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')
tr_set = set(y_train['custid'].values)
X = f503.fillna(0)
feat = list(X.columns)
print('503 로드:', X.shape)

# === attack(200) 피쳐파일 저장 ===
best_sel = json.load(open(f'{FEAT_DIR}/sel_attack.json'))
X[best_sel].to_parquet(f'{FEAT_DIR}/feats_attack200.parquet')
print(f'저장: feats_attack200.parquet ({len(best_sel)}개)')

# === B안 safe pool (te제외+KS<0.05) 전체 저장 ===
is_tr = pd.Series(X.index).isin(tr_set).values
ks_ser = pd.Series({
    c: (0.0 if (np.std(X[c].values[is_tr])<1e-9 and np.std(X[c].values[~is_tr])<1e-9)
        else ks_2samp(X[c].values[is_tr], X[c].values[~is_tr]).statistic)
    for c in feat
})
safe = [c for c in feat if ('te_' not in c) and (ks_ser[c] < 0.05)]
X[safe].to_parquet(f'{FEAT_DIR}/feats_safe498.parquet')
print(f'저장: feats_safe498.parquet ({len(safe)}개)')

print('\n저장 완료:')
print(f'  feats_attack200.parquet  — adversarial 안전, OOF 최적 200개')
print(f'  feats_safe498.parquet    — te제거+KS<0.05 클린 풀 498개')
print(f'  feats_safe_master172.parquet — 이미 있음 (도메인 정제 172개)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
503 로드: (49995, 503)
저장: feats_attack200.parquet (200개)
저장: feats_safe498.parquet (481개)

저장 완료:
  feats_attack200.parquet  — adversarial 안전, OOF 최적 200개
  feats_safe498.parquet    — te제거+KS<0.05 클린 풀 498개
  feats_safe_master172.parquet — 이미 있음 (도메인 정제 172개)


In [ ]:
#4. 200개 피처로 모델 앙상블 학습

from google.colab import drive
drive.mount('/content/drive')
!pip install catboost xgboost lightgbm -q

import pandas as pd, numpy as np, json
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

FEAT_DIR = '/content/drive/MyDrive/ML competition/feature'
DATA_DIR = '/content/drive/MyDrive/ML competition/competition_data'
CSV_DIR  = '/content/drive/MyDrive/ML competition/csv'

# === 데이터 ===
X_all = pd.read_parquet(f'{FEAT_DIR}/feats_attack200.parquet')
if 'custid' in X_all.columns: X_all = X_all.set_index('custid')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')
tr_ids = y_train['custid'].values; tr_set = set(tr_ids)
test_ids = [c for c in X_all.index if c not in tr_set]
t = y_train.set_index('custid')['gender'].loc[tr_ids].values
X_all = X_all.fillna(0)
Xtr = X_all.loc[tr_ids].values.astype(np.float32)
Xte = X_all.loc[test_ids].values.astype(np.float32)
print(f'200 피쳐 | train {len(tr_ids)} | test {len(test_ids)}')

# === OOF 검증 (5-fold) ===
skf = StratifiedKFold(5, shuffle=True, random_state=42)
folds = list(skf.split(Xtr, t))

def make_lgb(): return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03,
    num_leaves=31, subsample=0.8, colsample_bytree=0.55, reg_lambda=2.0,
    n_jobs=-1, max_bin=127, verbose=-1)
def make_xgb(): return xgb.XGBClassifier(n_estimators=400, learning_rate=0.03,
    max_depth=6, subsample=0.8, colsample_bytree=0.55, reg_lambda=2.0,
    n_jobs=-1, verbosity=0, eval_metric='auc')
def make_cat(): return CatBoostClassifier(iterations=400, learning_rate=0.04,
    depth=6, l2_leaf_reg=3, verbose=0)

# 각 모델: OOF + test 예측 (fold 평균)
def train_predict(make_fn):
    oof = np.zeros(len(Xtr)); test_pred = np.zeros(len(Xte))
    for tri, vi in folds:
        m = make_fn(); m.fit(Xtr[tri], t[tri])
        oof[vi] = m.predict_proba(Xtr[vi])[:, 1]
        test_pred += m.predict_proba(Xte)[:, 1] / len(folds)
    return oof, test_pred

print('LightGBM...'); o_lgb, p_lgb = train_predict(make_lgb)
print('XGBoost...');  o_xgb, p_xgb = train_predict(make_xgb)
print('CatBoost...'); o_cat, p_cat = train_predict(make_cat)

print(f'\nLightGBM OOF: {roc_auc_score(t, o_lgb):.5f}')
print(f'XGBoost  OOF: {roc_auc_score(t, o_xgb):.5f}')
print(f'CatBoost OOF: {roc_auc_score(t, o_cat):.5f}')

# === 블렌딩 (균등 평균) ===
oof_blend = (o_lgb + o_xgb + o_cat) / 3
test_blend = (p_lgb + p_xgb + p_cat) / 3
print(f'\n3모델 블렌딩 OOF: {roc_auc_score(t, oof_blend):.5f}')

# === 제출 저장 ===
pd.DataFrame({'custid': test_ids, 'gender': test_blend}).to_csv(
    f'{CSV_DIR}/submission_200_ensemble.csv', index=False)
print('saved submission_200_ensemble.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00
200 피쳐 | train 30000 | test 19995
LightGBM...
XGBoost...
CatBoost...

LightGBM OOF: 0.72119
XGBoost  OOF: 0.72118
CatBoost OOF: 0.72016

3모델 블렌딩 OOF: 0.72314
saved submission_200_ensemble.csv
